<a href="https://colab.research.google.com/github/gauravd12345/ml_proj/blob/main/ngram/ngram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [232]:
import re 
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

with open("/Users/gauravd/Desktop/ml_proj/datasets/hp_books1-7/hp_books/Book1.txt", "r", encoding="utf-8") as f:
    text = f.read()

""" seperating punctuation """
text = re.sub(r'([^\w\s])', r' \1 ', text)
text = re.sub(r'\s+', ' ', text).strip()

text = list(text.split())

In [233]:
""" hyperparameters """
vocab = list(set(text))
vocab_size = len(vocab)
embedding_dim = 100
window = 2 # trigram model

print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 6642


In [234]:
wtoi, itow = {"<>": 0}, {0: "<>"}
for i in range(len(vocab)):
    wtoi[vocab[i]] = i
    itow[i] = vocab[i]
    
X, y = [], []
for i in range(len(vocab) - window):
    xtok = text[i: i + window]
    ytok = [text[i + window]]
    
    X.append([wtoi[i] for i in xtok])
    y.append([wtoi[j] for j in ytok])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train, X_test = torch.tensor(X_train), torch.tensor(X_test)
y_train, y_test = torch.tensor(y_train).view(-1), torch.tensor(y_test).view(-1) # flattening to ensure softmax works

In [235]:
class nGram(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim)
        self.l1 = nn.Linear(window * embedding_dim, embedding_dim)
        self.l2 = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        logits = self.embed(x)
        logits = logits.view(len(x), -1) # flattening input vectors for softmax
        logits = self.l1(logits)
        logits = self.l2(logits)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nGram().to(device)
print(f"{model} \n Using {device} device")

nGram(
  (embed): Embedding(6642, 100)
  (l1): Linear(in_features=200, out_features=100, bias=True)
  (l2): Linear(in_features=100, out_features=6642, bias=True)
) 
 Using cpu device


In [236]:
epochs = 150
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

for epoch in range(epochs):
    optimizer.zero_grad()

    out = model(X_train)
    loss = criterion(out, y_train)

    loss.backward()
    optimizer.step()

    print(f"Epoch: {epoch + 1}, Loss: {loss:.2f}")

Epoch: 1, Loss: 8.84
Epoch: 2, Loss: 8.79
Epoch: 3, Loss: 8.74
Epoch: 4, Loss: 8.69
Epoch: 5, Loss: 8.63
Epoch: 6, Loss: 8.58
Epoch: 7, Loss: 8.52
Epoch: 8, Loss: 8.46
Epoch: 9, Loss: 8.40
Epoch: 10, Loss: 8.34
Epoch: 11, Loss: 8.27
Epoch: 12, Loss: 8.20
Epoch: 13, Loss: 8.13
Epoch: 14, Loss: 8.05
Epoch: 15, Loss: 7.97
Epoch: 16, Loss: 7.88
Epoch: 17, Loss: 7.79
Epoch: 18, Loss: 7.70
Epoch: 19, Loss: 7.60
Epoch: 20, Loss: 7.50
Epoch: 21, Loss: 7.39
Epoch: 22, Loss: 7.29
Epoch: 23, Loss: 7.17
Epoch: 24, Loss: 7.06
Epoch: 25, Loss: 6.95
Epoch: 26, Loss: 6.83
Epoch: 27, Loss: 6.72
Epoch: 28, Loss: 6.60
Epoch: 29, Loss: 6.49
Epoch: 30, Loss: 6.38
Epoch: 31, Loss: 6.27
Epoch: 32, Loss: 6.16
Epoch: 33, Loss: 6.06
Epoch: 34, Loss: 5.96
Epoch: 35, Loss: 5.86
Epoch: 36, Loss: 5.76
Epoch: 37, Loss: 5.67
Epoch: 38, Loss: 5.58
Epoch: 39, Loss: 5.50
Epoch: 40, Loss: 5.41
Epoch: 41, Loss: 5.33
Epoch: 42, Loss: 5.26
Epoch: 43, Loss: 5.18
Epoch: 44, Loss: 5.11
Epoch: 45, Loss: 5.04
Epoch: 46, Loss: 4.

In [244]:
import numpy as np

test_data = [[wtoi["THE"], wtoi["BOY"]]]
data = torch.tensor(test_data, dtype=torch.long).to(device)

length = 100
with torch.no_grad():
    for i in range(1, length):
        logits = model(data)
        pred = torch.argmax(logits, dim=1).item()
        print(itow[pred], end=" ")
        if i % 20 == 0:
            print("\n")
        test_data.append([test_data[-1][-1], pred])
        data = torch.tensor([test_data[-1]], dtype=torch.long).to(device)


WHO LIVED Mr . Dursley was enraged to see the bundle of blankets . Inside , and he slept . 

When he had been watching , all day , but they ’ ve got this son called Harry , but 

they ’ ve got this son called Harry , but they ’ ve got this son called Harry , but 

they ’ ve got this son called Harry , but they ’ ve got this son called Harry , but 

they ’ ve got this son called Harry , but they ’ ve got this son called Harry , 